# Code to reproduce predictions obtained from this model.

In [ ]:
import sys
sys.path.append('../../../code/loaders/')
sys.path.append('../../../code/modelling/')
sys.path.append('../../../code/helpers/')

### Dataset
This dataset (link) is already compiled. 

The dataset was created by hand labelling large DEM scenes, and exporting training tiles using ScaleNormalisedDataStack.py. See code/notebooks/colab/DataPreperation for an example of that.

In [ ]:
dataset_root = 'change directory root of google drive folder'

### Model

A pre-trained model is available at (link). To train a new model, with a specific region folded out, follow the cells below

In [ ]:
from MaskRCNN import MaskRCNN
from ModelCompare import train_for_deployment

model = MaskRCNN() # instantiates a new model
best_model = train_for_deployment(model, 'UK') # regions could be France, Russia, Karoo, Australia, Brazil, USA (used training regions)

To load a pretrained model instead, follow the cells below.

In [ ]:
from MaskRCNN import MaskRCNN

pretrained_state_path = 'path to state_dict'
state_dict = torch.load(pretrained_state_path)

model = MaskRCNN()
model.load_state_dict(state_dict)

### Inference

In [ ]:
# config, customisable

DEM_PATH = 'path'
RGB_PATH = 'path'
RESOLUTIONS = [1, 3, 6, 9, 12, 15]  # metres per pixel sweeps
OUT_PATH = 'path'

deployer = Deployer(
        dem_path=temp_path,
        rgb_path=rgb_path,
        model_state_dict=state_dict,
        veto_model_dict=veto_state_dict,
        device=device,
        resolutions=RESOLUTIONS,
        native_res = 1,
        veto_threshold=0.85)

# run inference
result = deployer.sweep(stride_frac=0.65)

# unpack outputs
(predictions, cleaned, rejected, veto_probability_map, out_transform, out_crs, support, coverage) = result

# merge predictions from different resolutions, and save shapefile.
deployer.merge_predictions(cleaned, out_transform, out_crs, output_path=OUT_PATH)